In [ ]:
#@title v-- Tap this if you play on Mobile { display-mode: "form" }
%%html
<b>Ensure this audio is playing to prevent colab from shutting down, then start KoboldCpp below</b><br/>
<audio autoplay="" src="https://raw.githubusercontent.com/KoboldAI/KoboldAI-Client/main/colab/silence.m4a" loop controls>


# KoboldCpp + Qwen3-VL-8B on free Colab (API-key secured)
Runtime > Change runtime type > **T4 GPU**, then run cells top to bottom.
Gives you an OpenAI-compatible `/v1` endpoint with vision behind a per-session API key.

**Security:** the Cloudflare URL is public, but generation requests require the API key printed by the launch cell. Keep that key private.


<a href="https://colab.research.google.com/github/LostRuins/koboldcpp/blob/concedo/colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi -L
!curl -fLo koboldcpp https://github.com/LostRuins/koboldcpp/releases/latest/download/koboldcpp-linux-x64 && chmod +x koboldcpp

In [ ]:
# Model (~4.5GB) + vision projector (mmproj). Check filenames on the HF repo if a download 404s.
BASE = "https://huggingface.co/unsloth/Qwen3-VL-8B-Instruct-GGUF/resolve/main"
!wget -q --show-progress -O model.gguf  {BASE}/Qwen3-VL-8B-Instruct-Q4_K_S.gguf
!wget -q --show-progress -O mmproj.gguf {BASE}/mmproj-F16.gguf

In [ ]:
# Launch in background with a public Cloudflare tunnel, protected by a random API key
import os, re, secrets, subprocess, time

# New strong key every Colab runtime. Keep it private.
KCPP_API_KEY = secrets.token_urlsafe(32)
env = os.environ.copy()
env["KCPP_PASSWORD"] = KCPP_API_KEY

cmd = [
    "./koboldcpp",
    "--model", "model.gguf",
    "--mmproj", "mmproj.gguf",
    "--usecuda",
    "--gpulayers", "999",
    "--contextsize", "16384",
    "--flashattention",
    "--port", "5001",
    "--remotetunnel",
    "--skiplauncher",
    "--maxrequestsize", "64",
    "--ratelimit", "1",
]

log_file = open("kcpp.log", "w")
proc = subprocess.Popen(cmd, stdout=log_file, stderr=subprocess.STDOUT, env=env)

url = None
for _ in range(90):
    time.sleep(4)
    try:
        log = open("kcpp.log", encoding="utf-8", errors="replace").read()
    except FileNotFoundError:
        continue
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", log)
    if m:
        url = m.group(0)
        break

print("Tunnel:", url)
print("OpenAI base URL:", (url or "see kcpp.log") + "/v1")
print("API key:", KCPP_API_KEY)
print("Use this same API key in Marine Pipeline Studio. Do not share it.")


In [ ]:
!tail -n 30 kcpp.log

In [ ]:
!pip -q install openai
from openai import OpenAI

# KoboldCpp --password/KCPP_PASSWORD is sent as an OpenAI Bearer API key.
c = OpenAI(base_url="http://localhost:5001/v1", api_key=KCPP_API_KEY)
r = c.chat.completions.create(
    model="koboldcpp",
    messages=[{"role":"user","content":"Say hi in one line."}],
)
print(r.choices[0].message.content)


In [ ]:
# Optional: verify the PUBLIC tunnel requires the API key
if url:
    remote = OpenAI(base_url=url + "/v1", api_key=KCPP_API_KEY)
    r = remote.chat.completions.create(
        model="koboldcpp",
        messages=[{"role":"user","content":"Reply exactly: AUTH OK"}],
        max_tokens=8,
    )
    print(r.choices[0].message.content)


In [ ]:
# Vision test (use your own image URL or base64 data URI)
import base64, requests
img = base64.b64encode(requests.get("https://upload.wikimedia.org/wikipedia/commons/thumb/4/47/PNG_transparency_demonstration_1.png/280px-PNG_transparency_demonstration_1.png").content).decode()
r = c.chat.completions.create(model="koboldcpp", messages=[{"role":"user","content":[
  {"type":"text","text":"Describe this image."},
  {"type":"image_url","image_url":{"url":"data:image/png;base64,"+img}}]}])
print(r.choices[0].message.content)